# YOLOv8 Fine-tuning & Deployment Workshop: Logistics Dataset

This repository contains the resolution of the practical workshop on fine-tuning and deploying a YOLOv8 model focused on object detection for logistics environments.

---

## Part 1: Data and Environment Preparation

For this project, the `logistics-sz9jr` (version 2) dataset hosted on Roboflow was used, which contains a high volume of images of warehouses, wood pallets, forklifts, and workers. 

**Ingestion Process:**
1. A cloud environment (Kaggle) was set up for heavy processing.
2. The `roboflow` Python library was used to authenticate via API and download the dataset directly in a YOLOv8-compatible format.
3. The main configuration file (`data.yaml`) maps the training and validation paths, and defines the logistics classes the model must learn to detect (e.g., `wood pallet`, `person`, `forklift`).

In [1]:
!pip install ultralytics roboflow

In [2]:
from roboflow import Roboflow
rf = Roboflow(api_key="sch31nWWIKHjtZwiOzAB")
project = rf.workspace("large-benchmark-datasets").project("logistics-sz9jr")
dataset = project.version(2).download("yolov8")

loading Roboflow workspace...
loading Roboflow project...


## Part 2: Model Training (Fine-tuning)

The fine-tuning process was executed in a cloud environment to efficiently handle the massive volume of data (66,211 training images and 18,988 validation images). 

**Training Configuration:**
* **Base Model:** YOLOv8n (`yolov8n.pt`). The "Nano" architecture was selected to prioritize inference speed for the local deployment while maintaining a highly competitive accuracy.
* **Hardware:** Kaggle Cloud environment accelerated via NVIDIA Tesla T4 GPU (15GB VRAM).
* **Hyperparameters:** * Epochs: 30
  * Image Size: 640x640
  * Batch Size: 16
  * Optimizer: Auto-selected `MuSGD` (lr=0.01, momentum=0.9)
* **Execution Time:** ~7.55 hours.

The model successfully converged over the 30 epochs without critical overfitting, automatically applying data augmentation techniques (Mosaic, blur, CLAHE) during the dataloader phase to improve generalization in diverse lighting conditions.

In [3]:
!yolo detect train model=yolov8n.pt data={dataset.location}/data.yaml epochs=30 imgsz=640 batch=16 project=runs/logistics name=y8n_colab

Ultralytics 8.4.36 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/Logistics-2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=y8n_colab2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100

In [ ]:
# evaluate
!yolo detect val model=/kaggle/working/runs/detect/runs/logistics/y8n_colab2/weights/best.pt data=/kaggle/working/Logistics-2/data.yaml

Ultralytics 8.4.36 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,009,548 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1417.9±586.9 MB/s, size: 42.3 KB)
val: Scanning /kaggle/working/Logistics-2/valid/labels.cache... 18988 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 18988/18988 2.7Git/s 0.0s
val: /kaggle/working/Logistics-2/valid/images/-Docker-1-Do_jpg.rf.5c00af35cf477fec60189674679ea214.jpg: 2 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1187/1187 8.3it/s 2:22<0.1s
                   all      18988      72322      0.765      0.671      0.729      0.527
               barcode        476        522      0.788      0.876      0.824      0.572
                   car       1953       2626       0.75      0.761       0.82      0.684
         cardboard box        837       9637      0.866      0.808     

In [8]:
# export
!yolo export model=/kaggle/working/runs/detect/runs/logistics/y8n_colab2/weights/best.pt format=onnx

Ultralytics 8.4.36 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
Model summary (fused): 73 layers, 3,009,548 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/kaggle/working/runs/detect/runs/logistics/y8n_colab2/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 24, 8400) (6.0 MB)

ONNX: starting export with onnx 1.20.1 opset 20...
ONNX: slimming with onnxslim 0.1.90...
ONNX: export success ✅ 1.1s, saved as '/kaggle/working/runs/detect/runs/logistics/y8n_colab2/weights/best.onnx' (11.7 MB)

Export complete (1.6s)
Results saved to /kaggle/working/runs/detect/runs/logistics/y8n_colab2/weights
Predict:         yolo predict task=detect model=/kaggle/working/runs/detect/runs/logistics/y8n_colab2/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/runs/detect/runs/logistics/y8n_colab2/weights/best.onnx imgsz=640 data=/kaggle/working/Logistics-2/data.yaml  
Visualize:       https://netron.app
💡 

## Part 3: Evaluation and Metrics

After training the YOLOv8n model for 30 epochs, the evaluation on the validation set (18,988 images) yielded highly satisfactory results, proving the model's viability for real-time logistics monitoring.

**Overall Performance:**
* **mAP50 (Mean Average Precision at IoU 0.5):** `72.9%`
* **mAP50-95:** `52.7%`
* **Precision:** `76.4%` (When the model detects an object, it is correct 76.4% of the time).
* **Recall:** `67.1%` (The model successfully identifies 67.1% of all ground-truth objects).

**Class-Level Analysis:**
The model demonstrated outstanding performance in detecting vehicles, safety gear, and structured markers:
* **Top Performers (mAP50 > 85%):** `truck` (92.2%), `forklift` (90.6%), `traffic light` (90.2%), `qr code` (88.4%), and `cardboard box` (87.6%). The structured nature of these objects makes their feature extraction highly efficient.
* **Areas for Improvement:** Performance drops on classes that are either highly amorphous or densely clustered. For instance, `fire` (46.6%) and `smoke` (64.4%) lack rigid bounding box features. Interestingly, `wood pallet` achieved a lower mAP50 (53.3%) and Recall (42.7%) despite having the highest instance count (17,374); this is a classic indicator of severe occlusion, visual blending with warehouse floors/racks, and dense clustering.

**Conclusion:**
The fine-tuned model provides a highly reliable baseline for core warehouse operations (tracking forklifts, trucks, and boxes). Future iterations could improve the detection of difficult classes (like wood pallets) by modifying the confidence thresholds, applying specialized data augmentation (like CutMix), or increasing the model size to YOLOv8s or YOLOv8m.

![Confusion Matrix](confusion_matrix.png)
![F1 curve](BoxF1_curve.png)
![Results](results.png)

## Part 4: Local API Deployment

For deploying the model into production, a **hybrid architecture** was chosen: training was performed in the cloud, but inference is executed on a local server to avoid latency issues and blocked ports.

**Deployment Process:**
1. **Optimization (ONNX):** The trained model was exported to the `.onnx` format (`best.onnx`). This allows for ultra-fast inferences on the CPU without relying on heavy NVIDIA/CUDA libraries in the local environment.
2. **Server (LitServe):** A RESTful API was implemented using `litserve` and `fastapi`. The server receives an image via a `multipart/form-data` request, processes it through the ONNX model, and returns the detections in JSON format.
3. **Clean Environment:** A native Windows dependency conflict was resolved by isolating the virtual environment with Python 3.11 and exclusively using the CPU versions of PyTorch (`torch`, `torchvision` --index-url cpu) along with `onnxruntime`.

**Inference Test:**
Below is a demonstration of the local server (port 8000) receiving a test image and returning the bounding box coordinates (`xyxy`) and confidence scores for the `wood pallet` class detections.


![Server](sever.png)

![cmd](cmd.png)